# Transcription autonome français–arabe

Ce notebook transcrit **un fichier audio ou vidéo** contenant principalement du français, avec des passages en arabe et des termes de fiqh. Il produit uniquement :

- un fichier `.txt` avec le texte transcrit ;
- un fichier `.srt` avec les horodatages ;
- une archive `.zip` contenant les deux résultats.

Il est totalement autonome : aucun dépôt Git ni token Hugging Face n'est nécessaire. Le fichier reste traité dans la session Colab.

> Avant de commencer : sélectionnez **Exécution → Modifier le type d'exécution → GPU** dans Colab.

Moteur utilisé : [Whisper large-v3](https://huggingface.co/openai/whisper-large-v3) avec [Faster-Whisper](https://github.com/SYSTRAN/faster-whisper).

## 1. Installer le moteur de transcription

L'installation et le premier téléchargement du modèle peuvent prendre quelques minutes.

In [ ]:
%pip -q install "faster-whisper==1.2.1"

print("Installation terminée.")

## 2. Régler la transcription

Le glossaire aide le modèle à reconnaître les termes spécialisés. Vous pouvez ajouter les noms de personnes, lieux, ouvrages et termes propres à votre enregistrement, dans l'orthographe souhaitée.

In [ ]:
# "upload" ouvre un sélecteur de fichier. "drive" lit un chemin Google Drive.
INPUT_MODE = "upload"
DRIVE_FILE = "/content/drive/MyDrive/mon_audio_ou_ma_video.mp4"

# large-v3 privilégie la qualité multilingue. Le modèle "turbo" est plus rapide,
# mais généralement moins prudent sur l'arabe et le vocabulaire spécialisé.
MODEL_SIZE = "large-v3"
BEAM_SIZE = 5
VAD_MIN_SILENCE_MS = 500

# Ajoutez ou retirez librement des variantes françaises, translittérées et arabes.
GLOSSARY = [
    "Allah", "الله",
    "Coran", "Qur'an", "القرآن",
    "Sunna", "Sunnah", "السنة",
    "hadith", "حديث",
    "fiqh", "فقه",
    "usul al-fiqh", "أصول الفقه",
    "madhhab", "مذهب",
    "charia", "sharia", "الشريعة",
    "halal", "حلال",
    "haram", "حرام",
    "wudu", "وضوء",
    "salat", "salah", "صلاة",
    "zakat", "زكاة",
    "aqida", "عقيدة",
    "tawhid", "توحيد",
    "fatwa", "فتوى",
    "ijtihad", "اجتهاد",
    "ijma", "إجماع",
    "qiyas", "قياس",
]

CONTEXT_PROMPT = (
    "Discussion principalement en français, avec des phrases en arabe et "
    "du vocabulaire islamique et de fiqh. Vocabulaire attendu : "
    + ", ".join(GLOSSARY)
    + "."
)

print(f"Modèle : {MODEL_SIZE}")
print(f"Glossaire : {len(GLOSSARY)} entrées")

## 3. Choisir l'audio ou la vidéo

Le mode `upload` convient aux fichiers raisonnablement petits. Pour un fichier volumineux, placez-le dans Google Drive, choisissez `INPUT_MODE = "drive"` ci-dessus et renseignez `DRIVE_FILE`.

In [ ]:
from pathlib import Path

if INPUT_MODE == "upload":
    from google.colab import files

    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Sélectionnez exactement un fichier audio ou vidéo.")
    input_path = Path(next(iter(uploaded))).resolve()
elif INPUT_MODE == "drive":
    from google.colab import drive

    drive.mount("/content/drive")
    input_path = Path(DRIVE_FILE).expanduser().resolve()
else:
    raise ValueError('INPUT_MODE doit être "upload" ou "drive".')

if not input_path.is_file():
    raise FileNotFoundError(f"Fichier introuvable : {input_path}")

size_mb = input_path.stat().st_size / (1024 * 1024)
print(f"Fichier : {input_path.name}")
print(f"Taille : {size_mb:.1f} Mo")

## 4. Charger Whisper

Le modèle `large-v3` est téléchargé une fois dans la session Colab. Un GPU est exigé ici, car son exécution sur CPU serait très lente.

In [ ]:
import torch
from faster_whisper import WhisperModel

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU non détecté. Dans Colab, choisissez Exécution → Modifier le type "
        "d'exécution → GPU, puis relancez toutes les cellules."
    )

gpu_name = torch.cuda.get_device_name(0)
print(f"GPU : {gpu_name}")
print("Chargement du modèle…")

model = WhisperModel(
    MODEL_SIZE,
    device="cuda",
    compute_type="float16",
    download_root="/content/whisper-models",
)

print("Modèle chargé.")

## 5. Transcrire

La détection multilingue est effectuée au fil des segments afin de mieux conserver les changements entre français et arabe. Le modèle transcrit sans traduire.

In [ ]:
from time import perf_counter

started_at = perf_counter()
hotwords = ", ".join(GLOSSARY)

segments_iterator, info = model.transcribe(
    str(input_path),
    task="transcribe",
    language=None,
    multilingual=True,
    beam_size=BEAM_SIZE,
    vad_filter=True,
    vad_parameters={
        "min_silence_duration_ms": VAD_MIN_SILENCE_MS,
        "speech_pad_ms": 250,
    },
    initial_prompt=CONTEXT_PROMPT,
    hotwords=hotwords,
    condition_on_previous_text=True,
    word_timestamps=False,
    language_detection_segments=3,
    language_detection_threshold=0.5,
    log_progress=True,
)

segments = []
for segment in segments_iterator:
    text = segment.text.strip()
    if not text or segment.end <= segment.start:
        continue
    item = {
        "start": float(segment.start),
        "end": float(segment.end),
        "text": text,
    }
    segments.append(item)
    print(f"[{item['start']:8.2f} → {item['end']:8.2f}] {text}")

elapsed = perf_counter() - started_at
if not segments:
    raise RuntimeError(
        "Aucune parole n'a été détectée. Vérifiez le fichier et le niveau audio."
    )

print()
print(f"Langue dominante détectée au départ : {info.language}")
print(f"Probabilité initiale : {info.language_probability:.1%}")
print(f"Segments conservés : {len(segments)}")
print(f"Temps de calcul : {elapsed / 60:.1f} min")

## 6. Créer et télécharger les résultats

In [ ]:
import re
from zipfile import ZIP_DEFLATED, ZipFile


def srt_timestamp(seconds: float) -> str:
    """Convertir un nombre de secondes au format HH:MM:SS,mmm."""
    total_ms = max(0, round(seconds * 1000))
    hours, remainder = divmod(total_ms, 3_600_000)
    minutes, remainder = divmod(remainder, 60_000)
    secs, milliseconds = divmod(remainder, 1000)
    return f"{hours:02}:{minutes:02}:{secs:02},{milliseconds:03}"


safe_stem = re.sub(r"[^\w.-]+", "_", input_path.stem, flags=re.UNICODE).strip("._")
safe_stem = safe_stem or "transcription"
output_dir = Path("/content/transcription-results")
output_dir.mkdir(parents=True, exist_ok=True)

txt_path = output_dir / f"{safe_stem}.transcription.txt"
srt_path = output_dir / f"{safe_stem}.transcription.srt"
zip_path = output_dir / f"{safe_stem}.transcription.zip"

plain_text = "\n".join(item["text"] for item in segments) + "\n"
txt_path.write_text(plain_text, encoding="utf-8")

srt_blocks = []
for index, item in enumerate(segments, start=1):
    srt_blocks.append(
        f"{index}\n"
        f"{srt_timestamp(item['start'])} --> {srt_timestamp(item['end'])}\n"
        f"{item['text']}"
    )
srt_path.write_text("\n\n".join(srt_blocks) + "\n", encoding="utf-8")

with ZipFile(zip_path, "w", compression=ZIP_DEFLATED) as archive:
    archive.write(txt_path, arcname=txt_path.name)
    archive.write(srt_path, arcname=srt_path.name)

print("Résultats créés :")
print(f"- {txt_path}")
print(f"- {srt_path}")
print(f"- {zip_path}")
print()
print("Aperçu :")
print(plain_text[:4000])

from google.colab import files

files.download(str(zip_path))

## Conseils de qualité

- Complétez `GLOSSARY` avec les noms propres et le vocabulaire réellement présents avant de transcrire.
- Écrivez chaque terme dans les variantes que vous souhaitez reconnaître : français, translittération et arabe.
- Un glossaire trop long ou rempli de termes absents peut provoquer de fausses reconnaissances : restez ciblé.
- Whisper peut parfois hésiter entre écriture arabe et translittération latine lors d'un changement de langue très court. Le SRT doit donc être relu avant publication.
- Pour un enregistrement difficile, conservez `large-v3` et utilisez un fichier source non compressé ou de bonne qualité.